# 02: Exploratory Data Analysis & Aggregation

## Objective
Aggregate individual admission records into daily time series and perform comprehensive exploratory data analysis.

## Tasks
- Aggregate individual records to daily admission counts (uses `aligned_date` from previous notebook if available)
- Create time series visualizations
- Analyze seasonality patterns
- Perform stationarity tests
- Document key patterns and insights

**Note:** This notebook automatically uses the `aligned_date` column (created in `01_data_extraction.ipynb` with hybrid day-of-week identification) for aggregation, which provides more accurate daily counts than deidentified dates.


## KPI Calculation

In addition to daily admissions, we'll also calculate related KPIs:
- Daily discharges
- Average length of stay (ALOS)
- Bed occupancy rates (if discharge data available)
- These KPIs can be used for VAR modeling and operational insights


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')
import os

# Setup paths
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hospital_admissions_forecasting'
    DRIVE_DATA_RAW = os.path.join(DRIVE_ROOT, 'data', 'raw')
    DRIVE_DATA_PROCESSED = os.path.join(DRIVE_ROOT, 'data', 'processed')
    DRIVE_RESULTS_PATH = os.path.join(DRIVE_ROOT, 'results', 'visualizations', 'eda_plots')
    for p in [DRIVE_DATA_RAW, DRIVE_DATA_PROCESSED, DRIVE_RESULTS_PATH]:
        os.makedirs(p, exist_ok=True)
except:
    DRIVE_DATA_RAW = DRIVE_DATA_PROCESSED = DRIVE_RESULTS_PATH = None

PROJECT_ROOT = '/content/hospital_admissions_forecasting'
DATA_RAW = os.path.join(PROJECT_ROOT, 'data', 'raw')
DATA_PROCESSED = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_PATH = os.path.join(PROJECT_ROOT, 'results', 'visualizations', 'eda_plots')
for p in [DATA_RAW, DATA_PROCESSED, RESULTS_PATH]:
    os.makedirs(p, exist_ok=True)

# Helper functions
def aggregate_to_daily(admissions_df, date_column='admittime'):
    """Aggregate individual admission records to daily counts.
    
    Uses 'aligned_date' column if available (already refined with day-of-week in 01_data_extraction.ipynb).
    Falls back to date_column if aligned_date is not available.
    """
    admissions_df = admissions_df.copy()

    # Prefer aligned_date (already refined with day-of-week identification in 01_data_extraction.ipynb)
    if 'aligned_date' in admissions_df.columns:
        use_column = 'aligned_date'
        print("  Using 'aligned_date' column (refined with day-of-week identification) for aggregation")
    else:
        use_column = date_column
        print(f"  Using '{date_column}' column (deidentified dates) for aggregation")

    admissions_df[use_column] = pd.to_datetime(admissions_df[use_column])
    daily_counts = admissions_df.groupby(admissions_df[use_column].dt.date).size()
    daily_counts.index = pd.to_datetime(daily_counts.index)
    date_range = pd.date_range(start=daily_counts.index.min(), end=daily_counts.index.max(), freq='D')
    return daily_counts.reindex(date_range, fill_value=0)

def calculate_daily_discharges(admissions_df, discharge_column='dischtime'):
    """Calculate daily discharge counts.

    Note: Discharge dates are not aligned to anchor year (only admission dates are aligned).
    This is acceptable since we're aggregating by discharge date, not aligning to admission anchor year.
    """
    if discharge_column not in admissions_df.columns:
        return pd.Series(dtype=int)
    admissions_df = admissions_df.copy()
    admissions_df[discharge_column] = pd.to_datetime(admissions_df[discharge_column])
    daily_discharges = admissions_df.groupby(admissions_df[discharge_column].dt.date).size()
    daily_discharges.index = pd.to_datetime(daily_discharges.index)
    if len(daily_discharges) > 0:
        date_range = pd.date_range(start=daily_discharges.index.min(), end=daily_discharges.index.max(), freq='D')
        daily_discharges = daily_discharges.reindex(date_range, fill_value=0)
    return daily_discharges

def calculate_average_length_of_stay(admissions_df, admission_column='admittime', discharge_column='dischtime'):
    """Calculate average length of stay (ALOS) per day."""
    if discharge_column not in admissions_df.columns:
        return pd.Series(dtype=float)
    admissions_df = admissions_df.copy()
    admissions_df[admission_column] = pd.to_datetime(admissions_df[admission_column])
    admissions_df[discharge_column] = pd.to_datetime(admissions_df[discharge_column])
    admissions_df['length_of_stay'] = (admissions_df[discharge_column] - admissions_df[admission_column]).dt.days
    daily_los = admissions_df.groupby(admissions_df[admission_column].dt.date)['length_of_stay'].mean()
    daily_los.index = pd.to_datetime(daily_los.index)
    return daily_los.sort_index()

def calculate_bed_occupancy(daily_admissions, daily_discharges, initial_beds=500, initial_occupancy=0.75):
    """Calculate daily bed occupancy rate."""
    start = min(daily_admissions.index.min(), daily_discharges.index.min() if len(daily_discharges) > 0 else daily_admissions.index.min())
    end = max(daily_admissions.index.max(), daily_discharges.index.max() if len(daily_discharges) > 0 else daily_admissions.index.max())
    date_range = pd.date_range(start=start, end=end, freq='D')

    daily_admissions = daily_admissions.reindex(date_range, fill_value=0)
    daily_discharges = daily_discharges.reindex(date_range, fill_value=0)
    bed_occupancy = pd.Series(index=date_range, dtype=int)
    bed_occupancy.iloc[0] = int(initial_beds * initial_occupancy)

    for i in range(1, len(date_range)):
        bed_occupancy.iloc[i] = max(0, min(bed_occupancy.iloc[i-1] + daily_admissions.iloc[i] - daily_discharges.iloc[i], initial_beds))
    return bed_occupancy, bed_occupancy / initial_beds

def create_kpi_dataframe(daily_admissions, daily_discharges=None, bed_occupancy=None, avg_los=None):
    """Create a DataFrame with multiple KPIs for VAR modeling.
    All KPIs are aligned to the same date range as daily_admissions."""
    # Use daily_admissions index as the master date range
    date_range = daily_admissions.index

    kpi_dict = {'admissions': daily_admissions}

    # Align all KPIs to the same date range
    if daily_discharges is not None and len(daily_discharges) > 0:
        kpi_dict['discharges'] = daily_discharges.reindex(date_range, fill_value=0)
    if bed_occupancy is not None and len(bed_occupancy) > 0:
        kpi_dict['bed_occupancy'] = bed_occupancy.reindex(date_range, method='ffill').fillna(0)
    if avg_los is not None and len(avg_los) > 0:
        kpi_dict['avg_length_of_stay'] = avg_los.reindex(date_range, method='ffill').fillna(0)

    kpi_df = pd.DataFrame(kpi_dict).sort_index()
    return kpi_df

def save_plot(filename, dpi=300):
    """Save plot to local and Google Drive."""
    local_path = os.path.join(RESULTS_PATH, filename)
    plt.savefig(local_path, dpi=dpi, bbox_inches='tight')
    if DRIVE_RESULTS_PATH:
        plt.savefig(os.path.join(DRIVE_RESULTS_PATH, filename), dpi=dpi, bbox_inches='tight')
        return local_path, os.path.join(DRIVE_RESULTS_PATH, filename)
    return local_path, None

# Set plotting style
for style in ['seaborn-v0_8-darkgrid', 'seaborn-darkgrid', 'seaborn', 'default']:
    try:
        plt.style.use(style)
        break
    except:
        pass



## Load Cleaned Data from Previous Notebook


In [ ]:
# Load cleaned admissions data
file_paths = [
    (DRIVE_DATA_RAW, 'admissions_clean.csv') if DRIVE_DATA_RAW else None,
    (DATA_RAW, 'admissions_clean.csv'),
    (DRIVE_DATA_RAW, 'admissions.csv') if DRIVE_DATA_RAW else None,
    (DATA_RAW, 'admissions.csv')
]

df = pd.DataFrame()
for folder, filename in [p for p in file_paths if p]:
    file_path = os.path.join(folder, filename)
    if os.path.exists(file_path):
        try:
            # Parse date columns (including aligned_date if present)
            date_cols = ['admittime', 'dischtime']
            df_temp = pd.read_csv(file_path, nrows=0)  # Read just headers
            if 'aligned_date' in df_temp.columns:
                date_cols.append('aligned_date')
            df = pd.read_csv(file_path, parse_dates=date_cols)
            print(f"✓ Loaded {len(df):,} records")
            break
        except:
            continue

if len(df) == 0:
    print("❌ Data file not found. Run notebook 01_data_extraction.ipynb first.")


## Aggregate to Daily Time Series


In [ ]:
# Aggregate to daily time series
if len(df) > 0:
    daily_admissions = aggregate_to_daily(df, date_column='admittime').to_frame(name='admissions')
    print(f"✓ Daily time series: {len(daily_admissions)} days | Mean: {daily_admissions['admissions'].mean():.2f} | Total: {daily_admissions['admissions'].sum():,}")
    
    # Save
    daily_admissions.to_csv(os.path.join(DATA_PROCESSED, 'daily_admissions.csv'))
    if DRIVE_DATA_PROCESSED:
        daily_admissions.to_csv(os.path.join(DRIVE_DATA_PROCESSED, 'daily_admissions.csv'))
else:
    daily_admissions = pd.DataFrame()


## Calculate KPIs


In [ ]:
# Calculate KPIs
if len(df) > 0 and 'dischtime' in df.columns:
    daily_discharges = calculate_daily_discharges(df).to_frame(name='discharges')
    avg_los = calculate_average_length_of_stay(df)
    if len(daily_discharges) > 0:
        bed_occupancy, bed_occupancy_rate = calculate_bed_occupancy(
            daily_admissions['admissions'], daily_discharges['discharges'], initial_beds=500, initial_occupancy=0.75
        )
        bed_occupancy_rate = bed_occupancy_rate.to_frame(name='occupancy_rate')
        print(f"✓ KPIs: {len(daily_admissions)} admissions, {len(daily_discharges)} discharges, {bed_occupancy_rate['occupancy_rate'].mean()*100:.1f}% avg occupancy")
    else:
        daily_discharges = pd.DataFrame()
        bed_occupancy_rate = pd.Series(dtype=float)
else:
    daily_discharges = pd.DataFrame()
    avg_los = pd.Series(dtype=float)
    bed_occupancy_rate = pd.Series(dtype=float)


## Distribution Analysis


In [ ]:
# Distribution analysis
if len(daily_admissions) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    data = daily_admissions['admissions']
    axes[0].hist(data, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.2f}')
    axes[0].axvline(data.median(), color='green', linestyle='--', label=f'Median: {data.median():.2f}')
    axes[0].set_title('Distribution of Daily Admissions')
    axes[0].set_xlabel('Number of Admissions')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].boxplot(data, vert=True)
    axes[1].set_title('Box Plot')
    axes[1].set_ylabel('Number of Admissions')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    save_plot('admissions_distribution.png')
    plt.show()


## Trend Analysis


In [ ]:
# Year-over-year and trend analysis
if len(daily_admissions) > 0:
    # Calculate yearly averages and counts
    yearly_avg = daily_admissions.groupby(daily_admissions.index.year)['admissions'].mean()
    yearly_count = daily_admissions.groupby(daily_admissions.index.year)['admissions'].count()
    yearly_total = daily_admissions.groupby(daily_admissions.index.year)['admissions'].sum()

    fig, axes = plt.subplots(2, 1, figsize=(15, 10))

    # Yearly trend
    axes[0].plot(yearly_avg.index, yearly_avg.values, marker='o', linewidth=2, markersize=8, color='steelblue')
    axes[0].set_title('Average Daily Admissions by Year', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Year', fontsize=12)
    axes[0].set_ylabel('Average Daily Admissions', fontsize=12)
    axes[0].grid(True, alpha=0.3)

    # Rolling 365-day average (yearly trend)
    if len(daily_admissions) >= 365:
        rolling_year = daily_admissions['admissions'].rolling(window=365, center=True).mean()
        axes[1].plot(daily_admissions.index, daily_admissions['admissions'], alpha=0.3, color='lightblue', label='Daily', linewidth=0.5)
        axes[1].plot(rolling_year.index, rolling_year.values, color='red', linewidth=2, label='365-day rolling average')
        axes[1].set_title('Daily Admissions with 365-Day Rolling Average (Trend)', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Date', fontsize=12)
        axes[1].set_ylabel('Number of Admissions', fontsize=12)
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_plot('trend_analysis.png')
    plt.show()


## KPI Time Series Visualization


In [ ]:
# Plot all KPIs together
if len(daily_admissions) > 0 and len(daily_discharges) > 0:
    fig, axes = plt.subplots(4, 1, figsize=(15, 12))

    # Admissions
    axes[0].plot(daily_admissions.index, daily_admissions['admissions'], alpha=0.6, color='steelblue', linewidth=0.5)
    axes[0].set_title('Daily Admissions', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Count', fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # Discharges
    if len(daily_discharges) > 0:
        axes[1].plot(daily_discharges.index, daily_discharges['discharges'], alpha=0.6, color='coral', linewidth=0.5)
        axes[1].set_title('Daily Discharges', fontsize=12, fontweight='bold')
        axes[1].set_ylabel('Count', fontsize=10)
        axes[1].grid(True, alpha=0.3)

    # Bed occupancy
    if isinstance(bed_occupancy_rate, pd.DataFrame):
        axes[2].plot(bed_occupancy_rate.index, bed_occupancy_rate['occupancy_rate'] * 100, alpha=0.6, color='green', linewidth=0.5)
        axes[2].set_title('Bed Occupancy Rate', fontsize=12, fontweight='bold')
        axes[2].set_ylabel('Percentage', fontsize=10)
        axes[2].grid(True, alpha=0.3)

    # Average LOS
    if len(avg_los) > 0:
        axes[3].plot(avg_los.index, avg_los.values, alpha=0.6, color='purple', linewidth=0.5)
        axes[3].set_title('Average Length of Stay', fontsize=12, fontweight='bold')
        axes[3].set_xlabel('Date', fontsize=10)
        axes[3].set_ylabel('Days', fontsize=10)
        axes[3].grid(True, alpha=0.3)

    plt.tight_layout()
    save_plot('kpi_timeseries.png')
    plt.show()


## Correlation Analysis


In [ ]:
# Correlation between KPIs
if len(daily_admissions) > 0 and len(daily_discharges) > 0 and len(avg_los) > 0:
    # Create correlation dataframe
    corr_data = {'admissions': daily_admissions['admissions']}
    if len(daily_discharges) > 0:
        corr_data['discharges'] = daily_discharges['discharges'].reindex(daily_admissions.index, fill_value=0)
    if isinstance(bed_occupancy_rate, pd.DataFrame):
        corr_data['bed_occupancy'] = bed_occupancy_rate['occupancy_rate'].reindex(daily_admissions.index, method='ffill').fillna(0)
    if len(avg_los) > 0:
        corr_data['avg_los'] = avg_los.reindex(daily_admissions.index, method='ffill').fillna(0)

    corr_df = pd.DataFrame(corr_data)
    corr_matrix = corr_df.corr()

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Matrix: KPIs')
    plt.tight_layout()
    save_plot('kpi_correlation.png')
    plt.show()
    print("Correlation matrix:\n", corr_matrix)


## Time Series Visualization


## Save KPI Data for VAR Model


In [ ]:
# Create and save KPI dataframe for VAR modeling
if len(daily_admissions) > 0 and len(daily_discharges) > 0 and len(avg_los) > 0:
    bed_occ = bed_occupancy_rate['occupancy_rate'] if isinstance(bed_occupancy_rate, pd.DataFrame) else bed_occupancy_rate
    kpi_df = create_kpi_dataframe(daily_admissions['admissions'], daily_discharges['discharges'], bed_occ, avg_los)
    kpi_df.to_csv(os.path.join(DATA_PROCESSED, 'daily_kpis.csv'))
    if DRIVE_DATA_PROCESSED:
        kpi_df.to_csv(os.path.join(DRIVE_DATA_PROCESSED, 'daily_kpis.csv'))
    print(f"✓ KPI dataframe saved: {kpi_df.shape} | Columns: {list(kpi_df.columns)}")


In [ ]:
# Plot full time series
if len(daily_admissions) > 0:
    plt.figure(figsize=(15, 6))
    plt.plot(daily_admissions.index, daily_admissions['admissions'], linewidth=0.5, alpha=0.7, color='steelblue')
    plt.title('Daily Hospital Admissions Over Time')
    plt.xlabel('Date')
    plt.ylabel('Number of Admissions')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    save_plot('daily_admissions_timeseries.png')
    plt.show()


## Time Series Decomposition


In [ ]:
# Time series decomposition
if len(daily_admissions) >= 365:
    try:
        decomposition = seasonal_decompose(daily_admissions['admissions'], model='additive', period=365, extrapolate_trend='freq')
        fig = decomposition.plot()
        fig.set_size_inches(15, 10)
        plt.suptitle('Time Series Decomposition', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        save_plot('decomposition.png')
        plt.show()
    except Exception as e:
        print(f"Decomposition failed: {e}")


## Seasonality Analysis - Day of Week Patterns


In [ ]:
# Day of week patterns
if len(daily_admissions) > 0:
    dow_avg = daily_admissions.groupby(daily_admissions.index.day_name())['admissions'].mean()
    dow_avg = dow_avg.reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
    plt.figure(figsize=(10, 6))
    dow_avg.plot(kind='bar', color='steelblue', edgecolor='black')
    plt.title('Average Daily Admissions by Day of Week')
    plt.xlabel('Day of Week')
    plt.ylabel('Average Admissions')
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    save_plot('day_of_week_pattern.png')
    plt.show()
    print(f"Day of week averages:\n{dow_avg}")


## Seasonality Analysis - Monthly Patterns


In [ ]:
# Monthly patterns
if len(daily_admissions) > 0:
    month_avg = daily_admissions.groupby(daily_admissions.index.month)['admissions'].mean()
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    plt.figure(figsize=(10, 6))
    plt.bar(range(1, 13), month_avg.values, color='coral', edgecolor='black')
    plt.xticks(range(1, 13), month_names, rotation=45)
    plt.title('Average Daily Admissions by Month')
    plt.xlabel('Month')
    plt.ylabel('Average Admissions')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    save_plot('monthly_pattern.png')
    plt.show()
    print(f"Monthly averages:\n{month_avg}")


## Stationarity Tests


In [ ]:
# Stationarity tests
if len(daily_admissions) > 0:
    adf_result = adfuller(daily_admissions['admissions'].dropna())
    print(f"ADF: Stat={adf_result[0]:.4f}, p={adf_result[1]:.4f} → {'STATIONARY' if adf_result[1] <= 0.05 else 'NON-STATIONARY'}")


In [ ]:
# KPSS test
if len(daily_admissions) > 0:
    try:
        kpss_result = kpss(daily_admissions['admissions'].dropna(), regression='ct')
        print(f"KPSS: Stat={kpss_result[0]:.4f}, p={kpss_result[1]:.4f} → {'STATIONARY' if kpss_result[1] >= 0.05 else 'NON-STATIONARY'}")
    except Exception as e:
        print(f"KPSS error: {e}")


## ACF and PACF Plots


In [ ]:
# ACF and PACF plots
if len(daily_admissions) > 0:
    data = daily_admissions['admissions'].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    plot_acf(data, lags=50, ax=axes[0], alpha=0.05)
    plot_pacf(data, lags=50, ax=axes[1], alpha=0.05, method='ols')
    axes[0].set_title('ACF')
    axes[1].set_title('PACF')
    plt.tight_layout()
    save_plot('acf_pacf.png')
    plt.show()
    print("ACF→MA(q), PACF→AR(p)")


## Summary Statistics


In [ ]:
# Summary statistics
if len(daily_admissions) > 0:
    stats = daily_admissions['admissions']
    peak_day = stats.idxmax()
    cv = stats.std() / stats.mean()
    print(f"Summary: {len(daily_admissions)} days | Mean: {stats.mean():.2f} | Std: {stats.std():.2f} | Range: {stats.min()}-{stats.max()}")
    print(f"Peak: {peak_day.strftime('%Y-%m-%d')} ({stats.max()} admissions) | CV: {cv:.2f}")


## Summary

This notebook:
- ✅ Aggregated individual admission records to daily time series (using `aligned_date` from previous notebook)
- ✅ Calculated KPIs (discharges, bed occupancy, ALOS) if data available
- ✅ Performed comprehensive exploratory data analysis
- ✅ Identified seasonal patterns (day of week, monthly)
- ✅ Tested for stationarity
- ✅ Generated ACF/PACF plots for model selection

**Key Findings:**
- [Document your key findings here after running]
- [Day of week patterns observed]
- [Seasonal trends identified]
- [Stationarity conclusion]

**Next Steps:** Proceed to `03_feature_engineering.ipynb` to create features for modeling.
